In [ ]:
````xml
<!-- filepath: /Users/bell/Programs/EcoFOCIpy/notebooks/04_Quality_Control.ipynb -->
<VSCode.Cell language="markdown">
# Quality Control Flagging and Data Validation

Master quality control (QC) procedures for oceanographic data, including flagging questionable measurements and identifying data anomalies.

**Topics covered:**
- QC flag standards and definitions
- Range checking and limit violations
- Spike detection and anomaly identification
- Rate of change checks
- Statistical outlier detection
- Visual QC assessment
- Creating QC reports

**Estimated time:** 25 minutes  
**Difficulty:** Intermediate to Advanced
</VSCode.Cell>
<VSCode.Cell language="python">
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("Quality Control Module")
print("="*60)
</VSCode.Cell>
<VSCode.Cell language="markdown">
## QC Flag Standards

Oceanographic data follow the **IODE (International Oceanographic Data Exchange)** QC flag convention:

| Flag | Status | Meaning | Action |
|------|--------|---------|--------|
| 1 | ✅ Good | Passed all QC checks | Use as-is |
| 2 | ✅ Good | Acceptable (alternate) | Use as-is |
| 3 | ⚠️ Questionable | Potentially problematic | Use with caution |
| 4 | ❌ Bad | Failed QC checks | Remove or interpolate |
| 9 | ❌ Missing | No data value | Skip/interpolate |

**Decision tree:**
```
Data point
  ├─ In valid range? 
  │   ├─ No → Flag = 4 (Bad)
  │   └─ Yes → Continue
  ├─ Passes spike check?
  │   ├─ No → Flag = 3 or 4
  │   └─ Yes → Continue
  ├─ Passes rate-of-change check?
  │   ├─ No → Flag = 3 or 4
  │   └─ Yes → Continue
  └─ Pass → Flag = 1 or 2 (Good)
```
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Step 1: Create Sample Data with Issues

Let's generate a realistic dataset containing common data quality problems.
</VSCode.Cell>
<VSCode.Cell language="python">
# Generate synthetic data with intentional quality issues
np.random.seed(42)
n_points = 500

# Base data - realistic oceanographic time series
time = pd.date_range('2024-01-01', periods=n_points, freq='15min')
base_temp = 10 + 5 * np.sin(np.arange(n_points) * 2*np.pi / (n_points/100))

# Add realistic noise
temperature = base_temp + np.random.normal(0, 0.3, n_points)

# Add quality issues:
# 1. Sensor drift at times 100-110
temperature[100:110] += 8

# 2. Spikes at random locations
spike_indices = np.random.choice(n_points, 10, replace=False)
temperature[spike_indices] += np.random.uniform(5, 10, 10)

# 3. Missing values represented as NaN
missing_indices = np.random.choice(n_points, 5, replace=False)
temperature[missing_indices] = np.nan

# 4. One sensor freeze period (constant values)
temperature[200:210] = temperature[199]

# Create DataFrame
data = pd.DataFrame({
    'time': time,
    'temperature': temperature.copy(),
    'salinity': 35 + 0.5*np.random.normal(0, 0.1, n_points)
})

print("SAMPLE DATA WITH QUALITY ISSUES")
print("="*60)
print(f"Total records: {len(data)}")
print(f"Date range: {data['time'].min()} to {data['time'].max()}")
print(f"Missing temperature values: {data['temperature'].isna().sum()}")
print(f"\nTemperature statistics:")
print(f"  Min: {data['temperature'].min():.2f} °C")
print(f"  Max: {data['temperature'].max():.2f} °C")
print(f"  Mean: {data['temperature'].mean():.2f} °C")
print(f"  Std: {data['temperature'].std():.2f} °C")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Step 2: Range Check

The first QC check is a simple range check - verify that values fall within acceptable limits.
</VSCode.Cell>
<VSCode.Cell language="python">
# Define acceptable ranges for oceanographic parameters
# These are typical ranges for the region being sampled
acceptable_ranges = {
    'temperature': (-3, 30),      # °C (typical ocean range)
    'salinity': (0, 40),          # PSU
}

# Initialize QC flags (1 = good by default)
qc_flags = pd.DataFrame(1, index=data.index, columns=['temperature', 'salinity'])

# Apply range checks
print("RANGE CHECK QC")
print("="*60)

for param, (min_val, max_val) in acceptable_ranges.items():
    # Find values outside acceptable range
    out_of_range = (data[param] < min_val) | (data[param] > max_val)
    qc_flags.loc[out_of_range, param] = 4  # Flag as bad
    
    # Also flag missing values
    qc_flags.loc[data[param].isna(), param] = 9  # Flag as missing
    
    print(f"\n{param.topper()}:")
    print(f"  Acceptable range: {min_val} to {max_val}")
    print(f"  Out of range values: {out_of_range.sum()}")
    print(f"  Missing values: {data[param].isna().sum()}")
    print(f"  QC Summary:")
    print(f"    Good (1):       {(qc_flags[param] == 1).sum()}")
    print(f"    Bad (4):        {(qc_flags[param] == 4).sum()}")
    print(f"    Missing (9):    {(qc_flags[param] == 9).sum()}")
</VSCode.Cell>
<VSCode.Cell language="python">
# Visualize range check results
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Temperature with range check highlighted
ax = axes[0]
ax.plot(data['time'], data['temperature'], 'b-', linewidth=1, label='Temperature data', alpha=0.7)

# Highlight flagged data
bad_range = qc_flags['temperature'] == 4
missing = qc_flags['temperature'] == 9

ax.scatter(data.loc[bad_range, 'time'], data.loc[bad_range, 'temperature'], 
          color='red', s=50, marker='x', label='Out of range', linewidths=2, zorder=10)
ax.scatter(data.loc[missing, 'time'], data.loc[missing, 'temperature'], 
          color='orange', s=50, marker='s', label='Missing', zorder=9)

# Show acceptable range as shaded area
ax.axhspan(-3, 30, alpha=0.1, color='green', label='Acceptable range')
ax.set_ylabel('Temperature (°C)', fontsize=11, fontweight='bold')
ax.set_title('Range Check QC - Temperature', fontsize=12, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.tick_params(axis='x', rotation=45)

# Summary bar chart
ax = axes[1]
categories = ['Good\n(1)', 'Bad\n(4)', 'Missing\n(9)']
counts = [
    (qc_flags['temperature'] == 1).sum(),
    (qc_flags['temperature'] == 4).sum(),
    (qc_flags['temperature'] == 9).sum()
]
colors_bar = ['green', 'red', 'orange']

bars = ax.bar(categories, counts, color=colors_bar, alpha=0.6, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Count', fontsize=11, fontweight='bold')
ax.set_title('QC Flag Distribution', fontsize=12, fontweight='bold')
ax.set_ylim(0, max(counts) * 1.1)

# Add count labels on bars
for bar, count in zip(bars, counts):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(count)}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Range check visualization complete")

In [ ]:
````xml
<!-- filepath: /Users/bell/Programs/EcoFOCIpy/notebooks/05_NetCDF_Export.ipynb -->
<VSCode.Cell language="markdown">
# NetCDF Export and File Format

Learn how to export oceanographic data to NetCDF format following CF (Climate and Forecast) conventions.

**Topics covered:**
- NetCDF file structure and concepts
- CF conventions for oceanographic data
- Creating self-documenting files
- Adding metadata and attributes
- Quality control flags in NetCDF
- Reading and verifying NetCDF files

**Estimated time:** 20 minutes  
**Difficulty:** Intermediate
</VSCode.Cell>
<VSCode.Cell language="python">
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    import xarray as xr
    import netCDF4 as nc
    print("✓ NetCDF libraries available")
except ImportError:
    print("⚠️  NetCDF libraries not installed")
    print("   Install with: pip install xarray netCDF4")

print("NetCDF Export Module")
print("="*60)
</VSCode.Cell>
<VSCode.Cell language="markdown">
## What is NetCDF?

**NetCDF** (Network Common Data Form) is a standardized file format for scientific data.

### Advantages:
- **Self-documenting**: Metadata stored with data
- **Efficient**: Compressed binary format reduces file size
- **Portable**: Works across platforms and languages
- **Standard**: Widely supported by oceanographic tools
- **CF-compliant**: Follows Climate and Forecast conventions

### File Structure:
```
NetCDF File
├── Dimensions
│   ├── time: 1000
│   ├── depth: 50
│   └── station: 10
├── Coordinates
│   ├── time [1000]
│   ├── depth [50]
│   ├── lon [10]
│   └── lat [10]
├── Data Variables
│   ├── temperature (time, depth) [1000, 50]
│   ├── salinity (time, depth) [1000, 50]
│   ├── temperature_QC (time, depth) [1000, 50]
│   └── salinity_QC (time, depth) [1000, 50]
└── Attributes
    ├── Global: title, institution, history
    └── Per-variable: units, long_name, valid_min/max
```
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Step 1: Prepare Data for Export

Organize your processed data into a format suitable for NetCDF export.
</VSCode.Cell>
<VSCode.Cell language="python">
# Create sample processed oceanographic data
print("Creating sample data for NetCDF export...")
print()

# Create time and depth dimensions
time = pd.date_range('2024-01-01', periods=100, freq='H')
depth = np.array([0, 10, 20, 50, 100, 200, 500])

# Create data arrays (time x depth)
np.random.seed(42)
temperature = 15 - 5*depth[np.newaxis,:]/100 + np.random.normal(0, 0.5, (len(time), len(depth)))
salinity = 35 + 0.5*depth[np.newaxis,:]/100 + np.random.normal(0, 0.1, (len(time), len(depth)))
oxygen = 400 - 100*depth[np.newaxis,:]/100 + np.random.normal(0, 5, (len(time), len(depth)))

# Create QC flags (1=good, 3=questionable, 4=bad)
temp_qc = np.ones((len(time), len(depth)), dtype=int)
sal_qc = np.ones((len(time), len(depth)), dtype=int)

# Add some flagged bad points
temp_qc[10:15, :] = 4
sal_qc[20:25, :] = 4

print(f"Data dimensions:")
print(f"  Time: {len(time)} records ({time[0]} to {time[-1]})")
print(f"  Depth: {len(depth)} levels ({depth[0]} to {depth[-1]} m)")
print(f"  Total points: {len(time) * len(depth)}")
print()
print(f"Temperature range: {temperature.min():.2f} to {temperature.max():.2f} °C")
print(f"Salinity range: {salinity.min():.2f} to {salinity.max():.2f} PSU")
print(f"Oxygen range: {oxygen.min():.0f} to {oxygen.max():.0f} µmol/kg")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Step 2: Create xarray Dataset

Convert data to xarray format, which makes NetCDF creation straightforward.
</VSCode.Cell>
<VSCode.Cell language="python">
try:
    import xarray as xr
    
    # Create coordinate arrays
    coords = {
        'time': time,
        'depth': depth,
        'lon': 175.5,  # Scalar coordinates
        'lat': 57.0
    }
    
    # Define dimension order
    dims = ['time', 'depth']
    
    # Create data arrays
    temp_da = xr.DataArray(temperature, coords={'time': time, 'depth': depth}, dims=dims)
    sal_da = xr.DataArray(salinity, coords={'time': time, 'depth': depth}, dims=dims)
    oxy_da = xr.DataArray(oxygen, coords={'time': time, 'depth': depth}, dims=dims)
    temp_qc_da = xr.DataArray(temp_qc, coords={'time': time, 'depth': depth}, dims=dims)
    sal_qc_da = xr.DataArray(sal_qc, coords={'time': time, 'depth': depth}, dims=dims)
    
    # Combine into dataset
    ds = xr.Dataset(
        data_vars={
            'temperature': temp_da,
            'salinity': sal_da,
            'oxygen': oxy_da,
            'temperature_QC': temp_qc_da,
            'salinity_QC': sal_qc_da
        },
        coords={
            'time': time,
            'depth': depth,
            'lon': 175.5,
            'lat': 57.0
        }
    )
    
    print("✓ xarray Dataset created successfully")
    print()
    print("Dataset structure:")
    print(ds)
    
except ImportError:
    print("⚠️  xarray not installed - skipping xarray examples")
    print("   Install with: pip install xarray")
except Exception as e:
    print(f"Error: {e}")